# Модуль-ноутбук: `train`

Baselines + LogReg + калібрований HGB, автовибір за OOF-Brier, тюнінг смуги рішення, персист `models/model.joblib` + метадані + звіти. `%run` підтягує `data_prep` (з `config/features/labels`) та `evaluate`.

**Залежності:** `%run` 03_data_prep.ipynb, 04_evaluate.ipynb

In [ ]:
%run 03_data_prep.ipynb
%run 04_evaluate.ipynb

In [ ]:
"""Train baselines + candidate models, evaluate head-to-head, auto-select the best-
calibrated deployable model, tune the decision band, and persist the artifact.

Run:  notebooks/05_train.ipynb
Outputs:
  models/model.joblib       deployable bundle (model + NN reference + scaler + explainer)
  models/metadata.json      full contract (features, thresholds, label def, metrics)
  models/reference.parquet  human-readable "similar past videos" reference table
  reports/metrics.json      all model metrics + CV diagnostics + A-vs-C demo
  reports/model_comparison.md, reports/plots/*.png
"""

import json

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import (
    GroupKFold, KFold, StratifiedKFold, cross_val_predict, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


SEED = config.RANDOM_SEED
# Transparent baseline on a few hand features: caption length, duration, time-of-day.
LOGREG3_FEATURES = ["char_len", "duration_s", "hour_sin", "hour_cos"]

In [ ]:
def make_hgb():
    return HistGradientBoostingClassifier(
        max_depth=3, max_iter=200, learning_rate=0.05, l2_regularization=1.0,
        min_samples_leaf=30, early_stopping=True, validation_fraction=0.15, random_state=SEED,
    )

In [ ]:
def make_logreg():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, C=0.5, random_state=SEED)),
    ])


# candidate factories eligible for deployment (interpretable LR vs calibrated boosting)

In [ ]:
def deploy_factory(name):
    if name == "logreg_full":
        return make_logreg()
    if name == "hgb":
        return CalibratedClassifierCV(make_hgb(), method="sigmoid", cv=5)
    raise ValueError(name)

In [ ]:
def cv_signal(raw_clean: pd.DataFrame, splitter, groups=None) -> dict:
    """Leakage-clean CV: re-fit label thresholds inside each fold; return AUC distribution.

    Diagnoses 'is there content signal at all', decoupled from the temporal drift that
    makes the headline holdout pessimistic.
    """
    aucs = []
    X_all = features.engineer_features(raw_clean)
    idx = np.arange(len(raw_clean))
    split_args = (idx, None, groups) if groups is not None else (idx, [0] * len(idx))
    for tr, va in splitter.split(*split_args):
        thr = labels.fit_creator_thresholds(raw_clean.iloc[tr])
        y = labels.make_labels(raw_clean, thr)
        ok_tr = tr[y.iloc[tr].notna().to_numpy()]
        ok_va = va[y.iloc[va].notna().to_numpy()]
        if len(np.unique(y.iloc[ok_va].astype(int))) < 2:
            continue
        model = make_hgb().fit(X_all.iloc[ok_tr], y.iloc[ok_tr].astype(int))
        p = model.predict_proba(X_all.iloc[ok_va])[:, 1]
        aucs.append(float(evaluate.roc_auc_score(y.iloc[ok_va].astype(int), p)))
    return {"mean_auc": float(np.mean(aucs)) if aucs else None,
            "std_auc": float(np.std(aucs)) if aucs else None,
            "folds": [round(a, 3) for a in aucs], "n_folds": len(aucs)}

In [ ]:
def main() -> None:
    config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    b = data_prep.prepare()
    Xtr, ytr, Xte, yte = b["X_train"], b["y_train"], b["X_test"], b["y_test"]
    print(f"train={len(Xtr)} (pos {ytr.mean():.3f})  test={len(Xte)} (pos {yte.mean():.3f})")

    # ---- test-set probabilities per model (head-to-head) ----
    probs = {}
    probs["majority"] = DummyClassifier(strategy="prior").fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    rate = ytr.groupby(b["train_df"][config.CREATOR_COL].values).mean()
    probs["creator_historical"] = b["test_df"][config.CREATOR_COL].map(rate).fillna(ytr.mean()).to_numpy()
    probs["logreg3"] = make_logreg().fit(Xtr[LOGREG3_FEATURES], ytr).predict_proba(Xte[LOGREG3_FEATURES])[:, 1]

    # ---- deployment candidates: OOF Brier (leakage-clean model selection) ----
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    oof, fitted, oof_brier = {}, {}, {}
    oof_brier_ci = {}
    for name in config.DEPLOY_CANDIDATES:
        oof[name] = cross_val_predict(deploy_factory(name), Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
        oof_brier[name] = float(brier_score_loss(ytr, oof[name]))
        oof_brier_ci[name] = evaluate.bootstrap_ci(
            ytr, oof[name], lambda y, p: float(brier_score_loss(y, p)))
        fitted[name] = deploy_factory(name).fit(Xtr, ytr)
        probs[name] = fitted[name].predict_proba(Xte)[:, 1]
    print("OOF Brier (lower=better calibrated):", {k: round(v, 4) for k, v in oof_brier.items()},
          "95% CI:", {k: [round(x, 3) for x in v] for k, v in oof_brier_ci.items()})

    deployed_name = (config.DEPLOY_MODEL if config.DEPLOY_MODEL in fitted
                     else min(oof_brier, key=oof_brier.get))
    deployed_model = fitted[deployed_name]
    explainer = "linear" if deployed_name == "logreg_full" else "agnostic"
    print(f"DEPLOYED: {deployed_name}  (selection: "
          f"{'forced' if config.DEPLOY_MODEL in fitted else 'min OOF Brier'})  explainer={explainer}")

    # ---- decision band tuned on the DEPLOYED model's TRAIN OOF probs ----
    (t_low, t_high), reached = evaluate.choose_thresholds(ytr, oof[deployed_name])
    print(f"decision band: t_low={t_low:.3f} t_high={t_high:.3f} (precision target reached: {reached})")

    # ---- metrics: ranking + decision @ 0.5 for the fair comparison table ----
    results = {name: evaluate.binary_metrics(yte, p, t_high=0.5) for name, p in probs.items()}
    deployed = evaluate.binary_metrics(yte, probs[deployed_name], t_high=t_high, t_low=t_low)
    # Brier floor: a constant predictor outputting the (unknowable-in-advance) test base rate.
    brier_floor = float(brier_score_loss(yte, np.full(len(yte), yte.mean())))
    ci_auc = evaluate.bootstrap_ci(yte, probs[deployed_name], evaluate._safe_auc)
    ci_ap = evaluate.bootstrap_ci(yte, probs[deployed_name],
                                  lambda y, p: float(evaluate.average_precision_score(y, p)))

    # ---- CV diagnostics (decoupled from drift) ----
    cv_random = cv_signal(b["train_df"], KFold(5, shuffle=True, random_state=SEED))
    groups = b["train_df"][config.CREATOR_COL].to_numpy()
    n_groups = len(np.unique(groups))
    cv_group = (cv_signal(b["train_df"], GroupKFold(n_splits=min(n_groups, 4)), groups=groups)
                if n_groups >= 2 else {"mean_auc": None, "note": "only 1 creator in train"})

    # ---- why C: label A vs C fame demo ----
    demo = evaluate.label_a_vs_c_demo(data_prep.clean(data_prep.load_raw()))

    # ---- global feature importance ----
    # Computed on a held-out slice carved from TRAIN (never the test set, which has ≈0.5 AUC
    # so permuting features there is pure noise). Leakage-clean: fit on 75% of train, permute
    # on the 25% it never saw. Reported with per-repeat std + a reliability caveat.
    Xi_tr, Xi_va, yi_tr, yi_va = train_test_split(
        Xtr, ytr, test_size=0.25, random_state=SEED, stratify=ytr)
    imp_model = deploy_factory(deployed_name).fit(Xi_tr, yi_tr)
    perm = permutation_importance(imp_model, Xi_va, yi_va, n_repeats=20,
                                  random_state=SEED, scoring="roc_auc")
    importance = sorted(
        ({"feature": f, "importance": float(mean), "std": float(sd)}
         for f, mean, sd in zip(features.FEATURE_COLUMNS, perm.importances_mean, perm.importances_std)),
        key=lambda d: d["importance"], reverse=True,
    )

    # ---- plots ----
    yte_np = yte.to_numpy()
    evaluate.plot_calibration({n: (yte_np, probs[n]) for n in config.DEPLOY_CANDIDATES},
                              config.PLOTS_DIR / "calibration.png")
    evaluate.plot_roc_pr({n: (yte_np, probs[n]) for n in
                          [*config.DEPLOY_CANDIDATES, "logreg3", "creator_historical"]},
                         config.PLOTS_DIR / "roc_pr.png")
    evaluate.plot_confusion(yte, probs[deployed_name], t_high, config.PLOTS_DIR / "confusion.png")

    # ---- persist deployable bundle ----
    Xtr_filled = Xtr.fillna(Xtr.median(numeric_only=True))
    nn_scaler = StandardScaler().fit(Xtr_filled)
    nn_matrix = nn_scaler.transform(Xtr_filled)
    er_tr = labels.compute_engagement_rate(b["train_df"])
    nn_meta = pd.DataFrame({
        "creator": b["train_df"][config.CREATOR_COL].values,
        "description": b["train_df"]["description"].astype("string").fillna("").values,
        "duration": pd.to_numeric(b["train_df"]["duration"], errors="coerce").values,
        "engagement_rate": er_tr.values,
        "play_count": pd.to_numeric(b["train_df"]["play_count"], errors="coerce").values,
        "label": ytr.values,
    })
    bundle = {
        "model": deployed_model,
        "deployed_name": deployed_name,
        "explainer": explainer,
        "feature_columns": features.FEATURE_COLUMNS,
        "feature_medians": Xtr.median(numeric_only=True).to_dict(),
        "decision_band": {"t_low": t_low, "t_high": t_high},
        "nn_scaler": nn_scaler,
        "nn_matrix": nn_matrix,
        "nn_meta": nn_meta.to_dict(orient="list"),
    }
    joblib.dump(bundle, config.MODEL_PATH)
    nn_meta.to_parquet(config.REFERENCE_PATH)

    metadata = {
        "model_version": "1.0.0",
        "deployed_model": deployed_name,
        "deployment_rationale": "best (lowest) train out-of-fold Brier among "
                                f"{config.DEPLOY_CANDIDATES}; honest probabilities are the "
                                "product's core value. OOF Brier=" + json.dumps({k: round(v, 4) for k, v in oof_brier.items()}),
        "label_definition": "success = engagement_rate > creator's TRAIN-median engagement_rate "
                            "(within-creator relative; ER excludes all-zero repost_count)",
        "feature_columns": features.FEATURE_COLUMNS,
        "decision_band": {"t_low": t_low, "t_high": t_high,
                          "precision_target": config.POST_PRECISION_TARGET,
                          "precision_target_reached": bool(reached)},
        "creator_thresholds": {"per_creator": b["thresholds"]["per_creator"],
                               "global_median": b["thresholds"]["global_median"]},
        "timezone_assumption": config.ASSUME_TIMEZONE,
        "split_meta": b["split_meta"],
        "test_metrics_deployed": deployed,
        "test_auc_95ci": ci_auc, "test_ap_95ci": ci_ap,
        "test_brier_floor_constant_baserate": brier_floor,
        "oof_brier": oof_brier, "oof_brier_95ci": oof_brier_ci,
        "top_features": importance[:10],
        "top_features_note": "permutation importance on a held-out TRAIN slice (test AUC≈0.5 "
                             "makes test-set importances unreliable); read with the std column.",
        "license": "Dataset: datahiveai/Tiktok-Videos, CC BY-NC 4.0 (research/non-commercial only)",
    }
    config.METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    metrics = {
        "comparison_decision_at_0.5": results,
        "deployed_model": deployed_name,
        "deployed_operating_point": deployed,
        "deployed_auc_95ci": ci_auc, "deployed_ap_95ci": ci_ap,
        "test_brier_floor_constant_baserate": brier_floor,
        "oof_brier": oof_brier, "oof_brier_95ci": oof_brier_ci,
        "cv_random_kfold_auc": cv_random,
        "cv_leave_one_creator_out_auc": cv_group,
        "label_A_vs_C_demo": demo,
        "top_features": importance,
        "split_meta": b["split_meta"],
    }
    config.METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    table = evaluate.comparison_markdown(results)
    (config.REPORTS_DIR / "model_comparison.md").write_text(
        f"# Model comparison (temporal test set, decision @ 0.5)\n\n{table}\n\n"
        f"- **Deployed: `{deployed_name}`** (lowest OOF Brier). OOF Brier: "
        f"{ {k: round(v,4) for k,v in oof_brier.items()} }\n"
        f"- Operating point: t_low={t_low:.3f}, t_high={t_high:.3f} "
        f"(precision target reached: {reached}); precision_post={deployed['precision_post']:.3f}, "
        f"recall_post={deployed['recall_post']:.3f}, "
        f"coverage Post/Unsure/No={deployed['frac_post']:.2f}/{deployed['frac_unsure']:.2f}/"
        f"{deployed['frac_dont']:.2f}\n"
        f"- ROC-AUC 95% CI: {ci_auc}; PR-AUC 95% CI: {ci_ap}\n"
        f"- Random 5-fold CV AUC (signal, no drift): **{cv_random['mean_auc']}**\n"
        f"- Leave-one-creator-out AUC: {cv_group.get('mean_auc')}\n"
        f"- Fame demo — creator-only AUC under label A: {demo['creator_only_auc_label_A']}, "
        f"under label C: {demo['creator_only_auc_label_C']}\n", encoding="utf-8")

    print("\n== comparison (decision @ 0.5) ==")
    print(table)
    print(f"\ndeployed={deployed_name} test AUC={deployed['roc_auc']} PR-AUC={deployed['pr_auc']} "
          f"Brier={deployed['brier']}")
    print(f"random-CV AUC={cv_random['mean_auc']}  LOCO-AUC={cv_group.get('mean_auc')}")
    print(f"A-vs-C creator-only AUC: {demo['creator_only_auc_label_A']} vs {demo['creator_only_auc_label_C']}")
    print(f"top features: {[d['feature'] for d in importance[:6]]}")
    print(f"saved: {config.MODEL_PATH.name}, {config.METADATA_PATH.name}, metrics.json")


if __name__ == "__main__":
    main()

In [ ]:
from types import SimpleNamespace
train = SimpleNamespace(
    make_hgb=make_hgb,
    make_logreg=make_logreg,
    deploy_factory=deploy_factory,
    cv_signal=cv_signal,
    main=main,
)

### Перевірка / демо

In [ ]:
train.main()   # навчає, оцінює, зберігає артефакти (~30–60 с)